# Energy Model Calibration (DESNZ vs ABM)

Self-contained calibration: load ABM outputs, load DESNZ LSOA elec+gas, compare per-dwelling (ABM per UPRN) to DESNZ per-meter, and plot distributions by year.


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

# display options
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 50)


## Paths and settings
Edit these as needed. `RUN_MODEL` defaults to False (use existing outputs).

In [ ]:
# Data paths (relative to repo root)
GEOJSON     = Path('data/epc_abm_newcastle.geojson')
CLIMATE     = Path('data/ncc_2t_timeseries_2010_2039.parquet')
HIDP_CSV    = Path('data/hidp_uprn_matches_tiered.csv')
ELEC_XLSX   = Path('data/LSOA_domestic_elec_2010-2023.xlsx')
GAS_XLSX    = Path('data/LSOA_domestic_gas_2010-2023.xlsx')
OUTDIR      = Path('results/calibration')
OUTDIR.mkdir(parents=True, exist_ok=True)

# Model outputs (produced separately via run.py)
MODEL_PARQUET = OUTDIR / 'model_timeseries.parquet'
AGENT_PARQUET = OUTDIR / 'agent_timeseries.parquet'

RUN_MODEL = False           # set True to rerun long model here (slow)
START_UTC = '2020-01-01T00:00:00Z'
YEARS = [2020, 2021, 2022, 2023]
LSOA_COL = 'lsoa_code'


## Load GeoJSON + HIDP enrichment (coalesce duplicate area codes)

In [ ]:
gdf = gpd.read_file(GEOJSON)
print(f"Loaded {len(gdf):,} dwellings from {GEOJSON}")

# ensure UPRN
if 'UPRN' not in gdf.columns:
    for alt in ['uprn','fid']:
        if alt in gdf.columns:
            gdf['UPRN'] = gdf[alt]
            break
assert 'UPRN' in gdf.columns, 'No UPRN field found.'
gdf['UPRN'] = gdf['UPRN'].astype(str).str.strip()

# HIDP merge (optional)
if HIDP_CSV.exists():
    hidp_df = pd.read_csv(HIDP_CSV, low_memory=False)
    hidp_df.columns = [c.strip() for c in hidp_df.columns]
    hidp_df['uprn_chr'] = hidp_df['uprn_chr'].astype(str).str.strip()
    hidp_df = hidp_df.drop_duplicates('uprn_chr')
    before = len(gdf)
    gdf = gdf.merge(hidp_df, how='left', left_on='UPRN', right_on='uprn_chr', suffixes=('_geo','_hidp'))
    print(f"Merged HIDP: {before:,} -> {len(gdf):,}; unmatched: {gdf['uprn_chr'].isna().sum():,}")
    # coalesce lsoa if duplicated
    if 'lsoa_code_geo' in gdf.columns or 'lsoa_code_hidp' in gdf.columns:
        gdf['lsoa_code'] = gdf.get('lsoa_code_geo').combine_first(gdf.get('lsoa_code_hidp'))
        gdf.drop(columns=[c for c in ['lsoa_code_geo','lsoa_code_hidp'] if c in gdf.columns], inplace=True)
else:
    print('Skipping HIDP enrichment (file not found)')

assert LSOA_COL in gdf.columns, f"{LSOA_COL} not found after merge"


## Optional: run model (slow) or load existing outputs
If `RUN_MODEL=False`, ensure `model_timeseries.parquet` and `agent_timeseries.parquet` exist in OUTDIR.


In [ ]:
from household_energy.model import EnergyModel
from household_energy.config import load_config

hours = len(YEARS) * 365 * 24

if RUN_MODEL:
    model = EnergyModel(
        gdf=gdf,
        climate_parquet=str(CLIMATE),
        climate_start=START_UTC,
        collect_agent_level=True,
        agent_collect_every=24,
    )
    for h in range(hours):
        model.step()
        if (h+1) % (365*24) == 0:
            print(f" progressed {h+1:,}/{hours:,} hours")
    model.model_dc.get_model_vars_dataframe().to_parquet(MODEL_PARQUET)
    model.agent_dc.get_agent_vars_dataframe().to_parquet(AGENT_PARQUET)
    print('Saved model & agent parquet')
else:
    assert MODEL_PARQUET.exists() and AGENT_PARQUET.exists(), 'Missing model/agent parquet; set RUN_MODEL=True to generate.'


## Load DESNZ LSOA data (elec + gas)

In [ ]:
import re

NORM_COLMAP = {
    "Local authority code": "la_code",
    "Local authority": "local_authority",
    "MSOA code": "msoa_code",
    "Middle layer super output area": "msoa_name",
    "LSOA code": "lsoa_code",
    "Lower layer super output area": "lsoa_name",
    "Number of meters": "meters",
    "Total consumption (kWh)": "total_kwh",
    "Mean consumption (kWh per meter)": "mean_kwh",
    "Median consumption (kWh per meter)": "median_kwh",
}

def tidy_lsoa(xlsx_path: Path, fuel: str) -> pd.DataFrame:
    book = pd.read_excel(xlsx_path, sheet_name=None, header=4, engine='openpyxl')
    # drop first sheet (notes)
    first = next(iter(book))
    book.pop(first, None)
    frames = []
    for sheet_name, df in book.items():
        df = df.copy()
        df.columns = df.columns.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
        df = df.rename(columns={k: v for k, v in NORM_COLMAP.items() if k in df.columns})
        wanted = ["lsoa_code","lsoa_name","meters","total_kwh"]
        keep = [c for c in wanted if c in df.columns]
        if not keep:
            continue
        df = df[keep].copy()
        for c in {"meters","total_kwh"} & set(keep):
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(',','',regex=False), errors='coerce')
        m = re.search(r"(20\d{2})", str(sheet_name))
        if not m:
            continue
        df["year"] = int(m.group(1))
        df["fuel"] = fuel
        frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["year","fuel","lsoa_code","meters","total_kwh"])
    out = pd.concat(frames, ignore_index=True)
    out = out[out["year"].isin(YEARS)]
    return out[["year","fuel","lsoa_code","meters","total_kwh"]]

desnz_elec = tidy_lsoa(ELEC_XLSX, "elec")
desnz_gas  = tidy_lsoa(GAS_XLSX,  "gas")
print(f"DESNZ elec rows: {len(desnz_elec):,}; gas rows: {len(desnz_gas):,}")


## ABM per-LSOA per-year (from agent parquet, no fallbacks)

In [ ]:
a_df = pd.read_parquet(AGENT_PARQUET)
hh_df = a_df[a_df['agent_type']=='household'].copy()
hh_df[LSOA_COL] = hh_df['AgentID'].astype(str).map(gdf.set_index('UPRN')[LSOA_COL].to_dict())
hh_df = hh_df.dropna(subset=[LSOA_COL])

hh_df['year'] = pd.to_datetime(hh_df['Step'], unit='h', origin=pd.Timestamp(START_UTC).tz_localize('UTC')).year
abm_totals = hh_df.groupby([LSOA_COL,'year'])['energy_consumption'].sum().reset_index().rename(columns={'energy_consumption':'abm_kwh'})

uprn_counts = gdf.groupby(LSOA_COL)['UPRN'].nunique().rename('uprn_count')
abm_totals['uprn_count'] = abm_totals[LSOA_COL].map(uprn_counts)
abm_totals['abm_kwh_per_dw'] = abm_totals['abm_kwh'] / abm_totals['uprn_count']


## DESNZ per-LSOA per-year (elec+gas / elec meters)

In [ ]:
desnz_long = pd.concat([desnz_elec, desnz_gas], ignore_index=True)

desnz_tot = (desnz_long.groupby([LSOA_COL,'year'])['total_kwh'].sum()
                        .reset_index().rename(columns={'total_kwh':'desnz_kwh'}))
elec_m = (desnz_elec.groupby([LSOA_COL,'year'])['meters'].sum()
                     .reset_index().rename(columns={'meters':'elec_meters'}))

desnz_tot = desnz_tot.merge(elec_m, on=[LSOA_COL,'year'], how='left')
desnz_tot['desnz_kwh_per_dw'] = desnz_tot['desnz_kwh'] / desnz_tot['elec_meters']


## Combine and analyze

In [ ]:
cmp = abm_totals.merge(desnz_tot, on=[LSOA_COL,'year'], how='inner')
cmp = cmp[cmp['year'].isin(YEARS)].copy()
cmp['ratio'] = cmp['abm_kwh_per_dw'] / cmp['desnz_kwh_per_dw']
print(f"Rows: {len(cmp):,} across {cmp[LSOA_COL].nunique()} LSOAs")

# Summary stats by year
summary = cmp.groupby('year').agg(
    n=('ratio','count'),
    mape=lambda s: (s.sub(1).abs()/1).mean()*100,
    median_ratio=('ratio','median'),
    mean_ratio=('ratio','mean')
)
display(summary.style.format({'mape':'{:.1f}%','median_ratio':'{:.3f}','mean_ratio':'{:.3f}'}).set_caption('ABM vs DESNZ per dwelling'))


## Plots: per-year scatter and ratio distribution

In [ ]:
years_sorted = sorted(cmp['year'].unique())
fig, axes = plt.subplots(1, len(years_sorted), figsize=(5*len(years_sorted),4), sharex=False, sharey=False)
if len(years_sorted)==1:
    axes=[axes]
for ax, yr in zip(axes, years_sorted):
    sub = cmp[cmp['year']==yr]
    ax.scatter(sub['desnz_kwh_per_dw'], sub['abm_kwh_per_dw'], alpha=0.5, s=18)
    mx = max(sub['desnz_kwh_per_dw'].max(), sub['abm_kwh_per_dw'].max())
    ax.plot([0,mx],[0,mx],'k--',lw=1)
    ax.set_title(f"{yr} (n={len(sub)})")
    ax.set_xlabel('DESNZ kWh per meter')
    ax.set_ylabel('ABM kWh per UPRN')
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
cmp.boxplot(column='ratio', by='year')
plt.axhline(1.0, color='k', linestyle='--', lw=1)
plt.ylabel('ABM / DESNZ (per dwelling)')
plt.title('ABM vs DESNZ ratio by year')
plt.suptitle(''); plt.tight_layout(); plt.show()
